# DR-ASPP-DRN: DME Fine-Tuning Pipeline
**End-to-end pipeline for DME risk grading on IDRiD dataset.**

Primary metric: **Quadratic Weighted Kappa (QWK) ≥ 0.80**

## 1. Environment Setup

In [ ]:
# Install dependencies (Kaggle / Colab)
# !pip install -r requirements.txt --quiet

import os, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available: {len(gpus)}')
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)

## 2. Configuration

In [ ]:
import yaml

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

# ── Paths (update as needed) ──────────────────────────────────────────────────
DATASET_PATH  = '/kaggle/input/idrid-disease-grading'  # or your local path
OUTPUT_DIR    = 'outputs'
BACKBONE_WEIGHTS = cfg['model'].get('backbone_weights', None)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config loaded:')
print(yaml.dump(cfg, default_flow_style=False))

## 3. Dataset Loading & EDA

In [ ]:
from dataset_loader import IDRiDDataset

train_dataset = IDRiDDataset(
    DATASET_PATH,
    split='train',
    augment=cfg['training']['augment'],
    batch_size=cfg['training']['batch_size'],
)
val_dataset = IDRiDDataset(
    DATASET_PATH,
    split='val',
    augment=False,
    batch_size=cfg['training']['batch_size'],
)

train_dataset.print_summary()
val_dataset.print_summary()

class_weights = train_dataset.get_class_weights()
print('Class weights:', class_weights)

## 4. Class Distribution Visualisation

In [ ]:
dist = train_dataset.class_distribution()
class_names = cfg['evaluation']['class_names']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(class_names, dist.values(), color=['#4CAF50', '#2196F3', '#FF5722'])
for bar, count in zip(bars, dist.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(count), ha='center', va='bottom', fontweight='bold')
ax.set_title('DME Grade Distribution (Train Split)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/class_distribution.png', dpi=150)
plt.show()

## 5. Preprocessing Verification

In [ ]:
from preprocess import preprocess_fundus_image, crop_black_borders, apply_clahe
import cv2

# Visualise preprocessing pipeline on a sample image
sample_path = train_dataset._image_paths[0]
raw = cv2.imread(sample_path)
raw = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)
cropped = crop_black_borders(raw)
enhanced = apply_clahe(cropped)
final = preprocess_fundus_image(sample_path)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ['Original', 'Border Cropped', 'CLAHE Enhanced', 'Final (512×512)']
images = [raw, cropped, enhanced, (final * 255).astype(np.uint8)]
for ax, img, title in zip(axes, images, titles):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/preprocessing_steps.png', dpi=150)
plt.show()
print(f'Final image shape: {final.shape}, range: [{final.min():.2f}, {final.max():.2f}]')

## 6. Model Architecture

In [ ]:
from model import build_model_dme_tuning

model = build_model_dme_tuning(
    backbone_weights_path=BACKBONE_WEIGHTS,
    freeze_backbone=cfg['model']['freeze_backbone'],
)

# Count parameters
total     = model.count_params()
trainable = sum(v.numpy().size for v in model.trainable_weights)
frozen    = total - trainable
print(f'Total params     : {total:,}')
print(f'Trainable params : {trainable:,}  ({100*trainable/total:.1f}%)')
print(f'Frozen params    : {frozen:,}  ({100*frozen/total:.1f}%)')

## 7. Build tf.data Pipelines

In [ ]:
train_ds = train_dataset.get_dataset()
val_ds   = val_dataset.get_dataset()

# Verify shapes
for images, labels in train_ds.take(1):
    print(f'Image batch shape : {images.shape}')
    print(f'Label batch shape : {labels.shape}')
    print(f'Image value range : [{images.numpy().min():.2f}, {images.numpy().max():.2f}]')

## 8. Compile Model

In [ ]:
from train import QuadraticWeightedKappa

lr = float(cfg['optimizer']['learning_rate'])
dme_w = float(cfg['loss']['dme_risk']['weight'])
dr_w  = float(cfg['loss']['dr_output']['weight'])

model.compile(
    optimizer=tf.keras.optimizers.Adam(lr),
    loss={'dr_output': 'mse', 'dme_risk': 'categorical_crossentropy'},
    loss_weights={'dr_output': dr_w, 'dme_risk': dme_w},
    metrics={
        'dr_output': [],
        'dme_risk': ['accuracy', QuadraticWeightedKappa(num_classes=3, name='qwk')],
    },
)
print('Model compiled ✅')

## 9. Training

In [ ]:
from train import QWKCallback
import time

qwk_cb = QWKCallback(val_ds, num_classes=3, log_file=f'{OUTPUT_DIR}/training_log.txt')

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_qwk', patience=cfg['training']['early_stopping']['patience'],
    mode='max', restore_best_weights=True, verbose=1,
)
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=f'{OUTPUT_DIR}/dme_finetuned.weights.h5',
    monitor='val_qwk', save_best_only=True, save_weights_only=True,
    mode='max', verbose=1,
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_qwk', factor=0.5, patience=3, mode='max', min_lr=1e-7, verbose=1,
)
tb = tf.keras.callbacks.TensorBoard(log_dir=f'{OUTPUT_DIR}/tb_logs')

print('Starting training …')
t0 = time.time()
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=cfg['training']['epochs'],
    callbacks=[qwk_cb, early_stop, checkpoint, reduce_lr, tb],
    class_weight=class_weights,
    verbose=1,
)
print(f'Training finished in {(time.time()-t0)/60:.1f} min')

## 10. Save Training History

In [ ]:
history_dict = {k: [float(v) for v in vals] for k, vals in history.history.items()}
history_dict['val_qwk'] = [e['val_qwk'] for e in qwk_cb.history]

with open(f'{OUTPUT_DIR}/training_history.json', 'w') as f:
    json.dump(history_dict, f, indent=2)
print('History saved ✅')

## 11. Training Curves

In [ ]:
from evaluate import plot_training_curves

plot_training_curves(history_dict, f'{OUTPUT_DIR}/training_curves.png')
plt.figure(figsize=(6, 4))
plt.imshow(plt.imread(f'{OUTPUT_DIR}/training_curves.png'))
plt.axis('off')
plt.show()

## 12. Evaluation

In [ ]:
from evaluate import DMEEvaluator

evaluator = DMEEvaluator(
    model, val_ds, OUTPUT_DIR,
    class_names=cfg['evaluation']['class_names'],
)
metrics = evaluator.run()

with open(f'{OUTPUT_DIR}/evaluation_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"\n{'='*50}")
print(f"  QWK      : {metrics['qwk']:.4f}")
print(f"  Accuracy : {metrics['accuracy']:.4f}")
print(f"  Macro F1 : {metrics['macro_f1']:.4f}")
print(f"{'='*50}")

## 13. Confusion Matrix

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(plt.imread(f'{OUTPUT_DIR}/confusion_matrix.png'))
plt.axis('off')
plt.title('DME Confusion Matrix')
plt.show()

## 14. Per-Class Analysis

In [ ]:
pc = metrics['per_class']
class_names = cfg['evaluation']['class_names']

print(f"{'Class':<22}{'Precision':>12}{'Recall':>10}{'F1':>10}{'Support':>10}")
print('-' * 64)
for i, name in enumerate(class_names):
    print(f"{name:<22}{pc['precision'][i]:>12.3f}{pc['recall'][i]:>10.3f}"
          f"{pc['f1'][i]:>10.3f}{pc['support'][i]:>10d}")

plt.figure(figsize=(7, 4))
plt.imshow(plt.imread(f'{OUTPUT_DIR}/per_class_metrics.png'))
plt.axis('off')
plt.show()

## 15. Comparison with Paper Benchmarks

In [ ]:
benchmarks = {
    'Li et al. (2019)':          0.82,
    'Ahammed et al. (2023)':     0.84,
    'Al-Antary et al. (2021)':   0.87,
    'Zhang et al. (2024)':       0.88,
    'DR-ASPP-DRN (ours)':        metrics['qwk'],
}

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['steelblue'] * (len(benchmarks) - 1) + ['crimson']
bars = ax.barh(list(benchmarks.keys()), list(benchmarks.values()), color=colors)
ax.axvline(0.80, color='green', linestyle='--', linewidth=1.5, label='Target ≥0.80')
for bar, val in zip(bars, benchmarks.values()):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)
ax.set_xlim(0.6, 1.0)
ax.set_xlabel('QWK Score')
ax.set_title('QWK Comparison with Literature Benchmarks')
ax.legend()
fig.tight_layout()
fig.savefig(f'{OUTPUT_DIR}/qwk_comparison.png', dpi=150)
plt.show()

## 16. Clinical Interpretation

In [ ]:
qwk = metrics['qwk']

interpretation = (
    '✅ EXCELLENT — Matches or exceeds state-of-the-art (≥0.85)'
    if qwk >= 0.85 else
    '✅ EXCELLENT — Meets clinical threshold (≥0.80)'
    if qwk >= 0.80 else
    '⚠️  GOOD — Close to target; consider unfreezing ASPP or more epochs'
    if qwk >= 0.70 else
    '❌ NEEDS IMPROVEMENT — Check class weights, learning rate, or data quality'
)

print(f'QWK = {qwk:.4f}')
print(interpretation)
print()
print('QWK interpretation scale:')
print('  < 0.20  Poor agreement')
print('  0.20–0.39  Fair')
print('  0.40–0.59  Moderate')
print('  0.60–0.79  Substantial')
print('  ≥ 0.80  Almost perfect (clinical grade)  ← TARGET')